# AI VERSE — CYRAN Worker (Kaggle)
**"One Universe. Infinite Intelligence."**

This notebook runs the official **CYRAN** advanced reasoning model worker on Kaggle GPU runtimes, powered by **GPT-OSS 20B**.
It implements the AI VERSE worker-pull protocol to securely claim high-priority queued jobs, perform 20B multi-step reasoning inference, and stream back results to the FastAPI backend.

**Important Architecture Rules:**
- Public Model Identifier: `cyran` (Clients interact exclusively with `cyran`; internal model names are never exposed to customers).
- Dedicated Platform: Kaggle.
- Worker ID: `cyran-kaggle-1`.

### Step 1: GPU Diagnostics & Hardware Compatibility Verification
Verifies GPU accelerators. GPT-OSS 20B requires ~11.5GB VRAM in 4-bit NF4 quantization or dual T4 (2x15GB = 30GB).

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Devices Available:", torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    raise SystemError("FATAL: No CUDA GPU detected! In Kaggle settings (right panel), select Accelerator: GPU T4 x 2 or GPU P100.")

total_available_vram = 0
for i in range(torch.cuda.device_count()):
    dev_vram = round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2)
    total_available_vram += dev_vram
    print(f"Device {i} [{torch.cuda.get_device_name(i)}]: {dev_vram} GB VRAM")

print(f"Aggregate Cluster VRAM: {total_available_vram:.2f} GB")

# Verification check for GPT-OSS 20B (~20B parameters)
if total_available_vram < 12.0:
    raise ValueError(f"Insufficient VRAM: GPT-OSS 20B requires at least 12GB VRAM (4-bit). Detected only {total_available_vram:.2f} GB.")
else:
    print("GPU Compatibility Verified: Hardware meets compute requirements for GPT-OSS 20B inference.")

### Step 2: Install Worker Dependencies
Installs `transformers`, `accelerate`, `bitsandbytes`, and `httpx`.

In [ ]:
!pip install -q --upgrade httpx transformers accelerate bitsandbytes sentencepiece

### Step 3: Load Kaggle Secrets & Configuration
Uses `kaggle_secrets.UserSecretsClient`:
- `AI_VERSE_API_URL`: Backend endpoint
- `CYRAN_WORKER_TOKEN`: Cryptographic worker token generated by Admin in AI VERSE Dashboard
- `HF_TOKEN` (Optional): Hugging Face token if using gated repositories

In [ ]:
import os
import sys

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    API_URL = user_secrets.get_secret("AI_VERSE_API_URL")
    WORKER_TOKEN = user_secrets.get_secret("CYRAN_WORKER_TOKEN")
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except Exception:
    API_URL = os.environ.get("AI_VERSE_API_URL", "http://127.0.0.1:8000")
    WORKER_TOKEN = os.environ.get("CYRAN_WORKER_TOKEN", "")
    HF_TOKEN = os.environ.get("HF_TOKEN")

WORKER_ID = "cyran-kaggle-1"
MODEL_ID = "CYRAN"  # Public model identifier: cyran
PROVIDER = "Kaggle"

# Exact model backing CYRAN
ACTUAL_MODEL_CHECKPOINT = os.environ.get("CYRAN_MODEL_CHECKPOINT", "openai/gpt-oss-20b")

if not WORKER_TOKEN:
    raise ValueError("CYRAN_WORKER_TOKEN secret is required! Add it under Kaggle Add-ons -> Secrets.")
if not API_URL:
    raise ValueError("AI_VERSE_API_URL secret is required!")

print(f"Configured Worker: {WORKER_ID} for Public Model: {MODEL_ID}")
print(f"Target API Gateway: {API_URL}")
print(f"Actual Model Architecture: GPT-OSS 20B ({ACTUAL_MODEL_CHECKPOINT})")

### Step 4: Modular CYRAN Model Inference Adapter (GPT-OSS 20B)
Loads GPT-OSS 20B using 4-bit NF4 quantization to maximize reasoning depth while maintaining high throughput on Kaggle dual T4 or P100 accelerators.

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

class CyranGptOssAdapter:
    """
    CYRAN Model Adapter powered by GPT-OSS 20B.
    Configures 4-bit NF4 quantization with bfloat16/float16 compute and automatic device mapping.
    """
    def __init__(self, model_checkpoint: str = ACTUAL_MODEL_CHECKPOINT, hf_token: str = HF_TOKEN):
        self.checkpoint = model_checkpoint
        self.token = hf_token
        self.tokenizer = None
        self.model = None
        self.generator = None

    def load_model(self):
        print(f"Loading CYRAN (GPT-OSS 20B) from {self.checkpoint}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.checkpoint, token=self.token)
        
        # 4-bit quantization config (fits 20B model into ~11.5GB VRAM)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        ) if torch.cuda.is_available() else None
        
        self.model = AutoModelForCausalLM.from_pretrained(
            self.checkpoint,
            token=self.token,
            quantization_config=bnb_config,
            device_map="auto" if torch.cuda.is_available() else None
        )
        self.generator = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer
        )
        print("CYRAN (GPT-OSS 20B) successfully initialized and mapped across GPU accelerators.")

    def generate(self, messages: list, max_new_tokens: int = 1536, temperature: float = 0.5) -> dict:
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        start_time = time.time()
        outputs = self.generator(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=max(temperature, 0.01),
            do_sample=True if temperature > 0.0 else False,
            pad_token_id=self.tokenizer.eos_token_id
        )
        latency = (time.time() - start_time) * 1000
        full_text = outputs[0]["generated_text"]
        reply = full_text[len(prompt):].strip()
        prompt_tokens = len(self.tokenizer.encode(prompt))
        completion_tokens = len(self.tokenizer.encode(reply))
        
        return {
            "text": reply,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "latency_ms": latency
        }

# Instantiate and load GPT-OSS 20B
cyran_engine = CyranGptOssAdapter()
cyran_engine.load_model()

### Step 5: Kaggle CYRAN Worker Daemon
Registers `cyran-kaggle-1`, issues periodic telemetry, and processes high-priority CYRAN reasoning jobs pulled from the queue.

In [ ]:
import httpx
import threading

client = httpx.Client(base_url=API_URL, timeout=50.0)
headers = {
    "X-Worker-Token": WORKER_TOKEN,
    "X-Worker-Id": WORKER_ID
}

# Registration
reg_payload = {
    "worker_id": WORKER_ID,
    "model_id": MODEL_ID,
    "provider": PROVIDER,
    "worker_version": "1.0.0"
}
reg_res = client.post("/api/workers/register", json=reg_payload, headers=headers)
print("Registration:", reg_res.status_code, reg_res.json())
if reg_res.status_code != 200:
    raise RuntimeError(f"Registration failed: {reg_res.text}")

# Heartbeat
stop_event = threading.Event()
current_job_holder = [None]

def heartbeat_loop():
    while not stop_event.is_set():
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        gpu_mem = int(torch.cuda.get_device_properties(0).total_memory / (1024**2)) if torch.cuda.is_available() else 0
        avail_mem = int(torch.cuda.memory_reserved(0) / (1024**2)) if torch.cuda.is_available() else 0
        status = "busy" if current_job_holder[0] else "online"
        
        payload = {
            "worker_id": WORKER_ID,
            "status": status,
            "gpu_name": gpu_name,
            "gpu_memory": gpu_mem,
            "available_memory": avail_mem,
            "current_job_id": current_job_holder[0]
        }
        try:
            client.post("/api/workers/heartbeat", json=payload, headers=headers)
        except Exception as e:
            print(f"Heartbeat error: {e}")
        time.sleep(15)

hb_thread = threading.Thread(target=heartbeat_loop, daemon=True)
hb_thread.start()

# Main Loop
print(f"[CYRAN WORKER ONLINE] Polling {MODEL_ID} queue from {API_URL}...")
try:
    while True:
        try:
            poll_res = client.get("/api/workers/jobs/next", headers=headers)
            if poll_res.status_code == 200:
                job_data = poll_res.json().get("job")
                if job_data:
                    job_id = job_data["job_id"]
                    current_job_holder[0] = job_id
                    print(f"\n[CYRAN CLAIMED] Job: {job_id} | Priority: {job_data.get('priority')}")
                    payload = job_data.get("payload", {})
                    messages = payload.get("messages", [])
                    temp = payload.get("temperature", 0.5)
                    max_tok = payload.get("max_tokens", 1536)
                    
                    try:
                        # Perform actual GPT-OSS 20B inference
                        gen_result = cyran_engine.generate(messages, max_new_tokens=max_tok, temperature=temp)
                        complete_payload = {
                            "result": {
                                "text": gen_result["text"],
                                "finish_reason": "stop",
                                "prompt_tokens": gen_result["prompt_tokens"],
                                "completion_tokens": gen_result["completion_tokens"]
                            },
                            "input_tokens": gen_result["prompt_tokens"],
                            "output_tokens": gen_result["completion_tokens"],
                            "latency_ms": gen_result["latency_ms"]
                        }
                        c_res = client.post(f"/api/workers/jobs/{job_id}/complete", json=complete_payload, headers=headers)
                        print(f"[CYRAN COMPLETE] Job {job_id} completed in {gen_result['latency_ms']:.1f}ms.")
                    except Exception as inf_err:
                        print(f"[ERROR] CYRAN inference failed: {inf_err}")
                        client.post(f"/api/workers/jobs/{job_id}/fail", json={"error": str(inf_err), "can_retry": True}, headers=headers)
                    
                    current_job_holder[0] = None
            time.sleep(1.5)
        except Exception as poll_err:
            print(f"Poll error: {poll_err}")
            time.sleep(3.0)
except KeyboardInterrupt:
    print("Stopping CYRAN worker...")
    stop_event.set()